# iML1515 Evaluation

The goal of this notebook is to evaluate iML1515, and create a nice overview of all the transport reactions, genes and substrates for futher analysis.

In [ ]:
import json
import csv

In order to only retain what iML1515 has deems transport related reactions, one needs to look into the subsystems of the reactions. I have set the transport reactions to be defined by a single keyword: By simply containing "transport" in the subsystem name.\
This was decided, as all subsystems that didn't contain that keyword, was not relevant for transport after inspecting all distinct subsystem values.

In [ ]:
with open("iML1515.json", "r") as f:
    model = json.load(f)

def is_transport_subsystem(subsystem):
    return "transport" in subsystem.lower()

# Create mappings for gene IDs to names, metabolite IDs to names, and a set of all genes in the model
gene_map = {g["id"]: g.get("name", "Unknown") for g in model["genes"]}
gene_uniprot_map = {g["id"]: g.get("annotation", {}).get("uniprot", ["Unknown"]) for g in model["genes"]}
gene_ids_in_model = {g["id"] for g in model["genes"]}
metabolite_map = {m["id"]: m["name"] for m in model["metabolites"]}
metabolite_chebi_map = {m["id"]: m["annotation"].get("chebi", ["Unknown"])[0] for m in model["metabolites"]}
metabolite_bigg_map = {m["id"]: m["annotation"].get("bigg.metabolite", ["Unknown"])[0] for m in model["metabolites"]}


# Store transport reactions and gene-tracking
transport_reactions = {}
all_genes = set()

# Iterate over all rxs in the model
for r in model["reactions"]:
    if "subsystem" in r and is_transport_subsystem(r["subsystem"]):
        metabolites = r.get("metabolites", {})

        # Extract gene IDs and UIDs
        gene_ids = r.get("gene_reaction_rule", "").split()
        gene_ids_clean = [g for g in gene_ids if g.strip("()") not in ["and", "or"]]
        genes = [gene_map.get(g.strip("()"), g.strip("()")) for g in gene_ids_clean if g.strip("()")]

        uniprot_ids = []
        for g in gene_ids_clean:
            g_clean = g.strip("()")
            if g_clean:
                uniprot_ids.extend(gene_uniprot_map.get(g_clean, ["Unknown"]))
        seen_uids = set()
        uniprot_ids = [x for x in uniprot_ids if not (x in seen_uids or seen_uids.add(x))]
        
        # Track all genes associated with the rx
        all_genes.update(genes)

        # Separate reactants and products for metabolite IDs
        reactants = [f"{-coef} {met}" for met, coef in metabolites.items() if coef < 0]
        products = [f"{coef} {met}" for met, coef in metabolites.items() if coef > 0]
        reaction_str = " + ".join(reactants) + " = " + " + ".join(products)

        # And metabolite names
        reactant_names = [f"{-coef} {metabolite_map.get(met, met)}" for met, coef in metabolites.items() if coef < 0]
        product_names = [f"{coef} {metabolite_map.get(met, met)}" for met, coef in metabolites.items() if coef > 0]
        reaction_str_names = " + ".join(reactant_names) + " = " + " + ".join(product_names)

        # And CHEBI names (only the first variant, thus not very useful for comparing reactions with the BLAST results)
        reactants_chebi = [f"{-coef} {metabolite_chebi_map.get(met, 'Unknown')}" for met, coef in metabolites.items() if coef < 0]
        products_chebi = [f"{coef} {metabolite_chebi_map.get(met, 'Unknown')}" for met, coef in metabolites.items() if coef > 0]
        reaction_str_chebi = " + ".join(reactants_chebi) + " = " + " + ".join(products_chebi)

        # And the set of substrates on the form of BiGG IDs
        substrates_bigg_set = {metabolite_bigg_map.get(met, "Unknown") for met, _ in metabolites.items()}

        # Store details on rx
        transport_reactions[r["id"]] = {
            "name": r["name"],
            "subsystem": r.get("subsystem", "Unknown"),
            "reaction": reaction_str,
            "reaction_names": reaction_str_names,
            "reaction_chebi": reaction_str_chebi,
            "substrates_bigg": substrates_bigg_set,
            "genes": genes if genes else ["Unknown"],
            "uniprot_ids": uniprot_ids if uniprot_ids else ["Unknown"]
        }

print("Quick findings on transporters in iML1515:")
print(f"{len(transport_reactions)} transport reactions")
print(f"These reactions are coded for by {len(all_genes)-1} different genes in the model, when ignoring 'Unknown' genes")

def save_tsv(fname, df):
    with open(fname, "w", newline="") as f:
        writer = csv.writer(f, delimiter="\t")
        writer.writerow(["Reaction ID", "Name", "Subsystem", "Reaction (IDs)", "Reaction (Names)", "Reaction (CHEBI)", "Substrates (BiGG)", "Genes", "UIDs"])
        
        for rid, data in df.items():
            writer.writerow([rid, data["name"], data["subsystem"], data["reaction"], data["reaction_names"], data["reaction_chebi"], data["substrates_bigg"],", ".join(data["genes"]), ", ".join(data["uniprot_ids"])])

save_tsv("iML1515_transport_rxs.tsv", transport_reactions)

Quick findings on transporters in iML1515:
817 transport reactions
These reactions are coded for by 405 different genes in the model, when ignoring 'Unknown' genes
